In [1]:
import numpy as np

In [2]:
# alpha = np.deg2rad(-30) # degrees
# torque_max = 5
# length = 1
# mass = 1
# gravity = 9.81


# alpha_double_prime = ((gravity / length) *np.cos(alpha)) + (torque / (mass * length**2))
# alpha_prime = ((gravity / length) *np.sin(alpha)) +  (torque / (mass * length**2)) * alpha + 0

# reward = -10 * (alpha - 90)**2 - 1 * (alpha_prime**2) - 0.0001 * torque

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class Penv(gym.Env):
    def __init__(self):
        super(Penv, self).__init__()
        
        # Constants
        self.length = 1.0
        self.mass = 1.0
        self.gravity = 9.81
        self.torque_max = 2.0
        self.dt = 0.1
        
        # Action = torque
        self.action_space = spaces.Box(
            low=-self.torque_max,
            high=self.torque_max,
            shape=(1,),
            dtype=np.float32
        )
        
        # Observation = [alpha, alpha_dot]
        self.observation_space = spaces.Box(
            low=np.array([-np.pi, -8]),
            high=np.array([np.pi, 8]),
            dtype=np.float32
        )
        
        self.reset()

    def step(self, action):
        torque = np.clip(action[0], -self.torque_max, self.torque_max)
        
        alpha, alpha_dot = self.state
        
        # Dynamics
        alpha_ddot = (
            (self.gravity / self.length) * np.sin(alpha)
            + torque / (self.mass * self.length**2)
        )
        
        alpha_dot += alpha_ddot * self.dt
        alpha += alpha_dot * self.dt
        
        self.state = np.array([alpha, alpha_dot])
        
        # Target: upright (90 degrees = pi/2)
        target = np.pi / 2
        
        reward = (
            -10 * (alpha - target) ** 2
            - 1 * alpha_dot**2
            - 0.0001 * abs(torque)
        )
        terminated=False
        if abs(alpha - target) < 0.1 and abs(alpha_dot) < 0.1:
            reward+=40
            terminated = True
        
        return self.state, reward, terminated, False, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        alpha = np.random.uniform(-np.pi, 0)
        alpha_dot = 0.0
        
        self.state = np.array([alpha, alpha_dot], dtype=np.float32)
        return self.state, {}

In [4]:
from stable_baselines3 import PPO

env = Penv()

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
)

model.learn(total_timesteps=100_000)

model.save("ppo_pendulum")

/workspaces/CSCI_RL/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/workspaces/CSCI_RL/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 1590 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 1223         |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 6.864243e-05 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -0.00033     |
|    learning_rate        | 0.0001       |
|    loss                 | 8.06e+06     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.000121    |
|    std

In [8]:
import numpy as np
import plotly.graph_objects as go

# Initialize environment and model
env = Penv()
model = PPO.load("ppo_pendulum")

obs, _ = env.reset()

# Storage
x_vals, y_vals, torque_vals = [], [], []

steps = 300

for _ in range(steps):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env.step(action)
    
    theta = obs[0]
    x = np.sin(theta)
    y = -np.cos(theta)
    
    x_vals.append(x)
    y_vals.append(y)
    torque_vals.append(action[0])  # Assuming continuous 1D torque

    if terminated or truncated:
        obs, _ = env.reset()

# Create figure with subplots (pendulum + torque)
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, 
                    column_widths=[0.6, 0.4],
                    specs=[[{"type":"scatter"}, {"type":"scatter"}]],
                    subplot_titles=("Pendulum", "Torque"))

# Initial pendulum frame
pendulum = go.Scatter(x=[0, x_vals[0]], y=[0, y_vals[0]],
                      mode='lines+markers', line=dict(width=4, color='blue'),
                      marker=dict(size=12, color='red'), name='Pendulum')
# Initial torque frame
torque_line = go.Scatter(x=[0], y=[torque_vals[0]],
                         mode='lines+markers', line=dict(color='green', width=3),
                         marker=dict(size=6), name='Torque')

fig.add_trace(pendulum, row=1, col=1)
fig.add_trace(torque_line, row=1, col=2)

# Frames
frames = []
for k in range(steps):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[0, x_vals[k]], y=[0, y_vals[k]],
                       mode='lines+markers', line=dict(width=4, color='blue'),
                       marker=dict(size=12, color='red')),
            go.Scatter(x=list(range(k+1)), y=torque_vals[:k+1],
                       mode='lines+markers', line=dict(color='green', width=3),
                       marker=dict(size=6))
        ]
    ))

# Layout
fig.update_layout(
    width=900, height=500,
    yaxis=dict(range=[-1.2, 1.2], scaleanchor="x"),
    yaxis2=dict(range=[-3, 3]),  # torque range (adjust as needed)
    xaxis2=dict(title="Step"),
    title="PPO Pendulum with Torque",
    updatemenus=[dict(
        type="buttons",
        buttons=[dict(label="Play",
                      method="animate",
                      args=[None, {"frame": {"duration": 20, "redraw": True},
                                   "fromcurrent": True}]),
                 dict(label="Pause",
                      method="animate",
                      args=[[None], {"frame": {"duration": 0, "redraw": False},
                                     "mode": "immediate"}])]
    )]
)

fig.frames = frames

fig.show()